In [2]:
from ollama import chat
LLM_MODEL = "qwen2.5-coder:3b"
response = chat(
        model=LLM_MODEL,
        messages=[
            {
                "role": "user",
                "content": "hi"
            }
        ]
    )

response

ChatResponse(model='qwen2.5-coder:3b', created_at='2026-09-09T05:27:26.812718157Z', done=True, done_reason='stop', total_duration=9053169195, load_duration=6663377003, prompt_eval_count=30, prompt_eval_duration=910432364, eval_count=22, eval_duration=1402673832, message=Message(role='assistant', content="Hello! How can I assist you today? Is there anything specific you'd like to know or discuss?", thinking=None, images=None, tool_name=None, tool_calls=None), logprobs=None)

In [6]:
import torch
import sounddevice as sd
import numpy as np
from scipy.io.wavfile import write
from pathlib import Path
import subprocess
import time
from transformers import AutoTokenizer
import onnxruntime as ort
from ollama import chat
from kokoro import KPipeline
from IPython.display import Audio, display
import soundfile as sf

# ============================================================
# Configuration
# ============================================================

SAMPLE_RATE = 16000
CHUNK_SIZE = 512

SPEECH_THRESHOLD = 0.5
SILENCE_DURATION = 1.0
MAX_UTTERANCE_SECONDS = 30

LLM_MODEL = "qwen2.5-coder:3b"

WHISPER_DIR = Path("../whisper.cpp")
WHISPER_MODEL = WHISPER_DIR / "models" / "ggml-base.en.bin"
WHISPER_CLI = WHISPER_DIR / "build" / "bin" / "whisper-cli"

AUDIO_FILE = Path("utterance.wav")

SHOULD_RESPOND_CLASS = 1

VOICE = "af_heart"


# ============================================================
# Load Should AI Respond model
# ============================================================

model, utils = torch.hub.load(
    repo_or_dir="snakers4/silero-vad",
    model="silero_vad",
    trust_repo=True
)

tokenizer = AutoTokenizer.from_pretrained(
    "./should_ai_respond_model"
)

print("Loaded tokenizer")


int8_session = ort.InferenceSession(
    "./should_ai_respond_int8.onnx",
    providers=["CPUExecutionProvider"]
)

print("Loaded INT8 BERT classification model")

pipeline = KPipeline(lang_code="a")

stream = sd.OutputStream(
    samplerate=24000,
    channels=1,
    dtype="float32",
    blocksize=2048
)


# ============================================================
# Helper
# ============================================================

def softmax(x):
    exp_x = np.exp(
        x - np.max(x, axis=1, keepdims=True)
    )
    return exp_x / exp_x.sum(
        axis=1,
        keepdims=True
    )


# ============================================================
# Main loop
# ============================================================

print("Listening...")
print("Speak now.\n")


while True:

    # Reset state for every utterance

    audio_chunks = []
    speech_started = False
    silence_start = None

    utterance_start_time = time.time()

    model.reset_states()

    with sd.InputStream(
        samplerate=SAMPLE_RATE,
        channels=1,
        dtype="float32",
        blocksize=CHUNK_SIZE
    ) as stream:

        while True:

            chunk, overflowed = stream.read(CHUNK_SIZE)

            chunk = chunk[:, 0]

            chunk_tensor = torch.from_numpy(chunk)

            speech_probability = model(
                chunk_tensor,
                SAMPLE_RATE
            ).item()

            is_speech = (
                speech_probability >= SPEECH_THRESHOLD
            )

            # ----------------------------------------
            # Speech
            # ----------------------------------------

            if is_speech:

                if not speech_started:
                    print("Speech detected...")
                    speech_started = True

                audio_chunks.append(chunk.copy())

                silence_start = None

            # ----------------------------------------
            # Silence
            # ----------------------------------------

            else:

                if speech_started:

                    audio_chunks.append(chunk.copy())

                    if silence_start is None:
                        silence_start = time.time()

                    silence_time = (
                        time.time() - silence_start
                    )

                    if silence_time >= SILENCE_DURATION:
                        print("User finished speaking.")
                        break

            # ----------------------------------------
            # Maximum utterance duration
            # ----------------------------------------

            if (
                time.time() - utterance_start_time
                >= MAX_UTTERANCE_SECONDS
            ):
                print("Maximum utterance duration reached.")
                break


    # ========================================================
    # No speech
    # ========================================================

    if not speech_started:
        print("No speech detected.")
        continue


    # ========================================================
    # Combine audio
    # ========================================================

    audio = np.concatenate(audio_chunks)

    write(
        AUDIO_FILE,
        SAMPLE_RATE,
        audio
    )


    # ========================================================
    # Whisper
    # ========================================================

    print("Calling Whisper...")

    try:

        result = subprocess.run(
            [
                str(WHISPER_CLI),
                "-m", str(WHISPER_MODEL),
                "-f", str(AUDIO_FILE),
                "-nt"
            ],
            capture_output=True,
            text=True,
            check=True
        )

    except subprocess.CalledProcessError as e:

        print("Whisper failed:")
        print(e.stderr)

        continue


    user_text = result.stdout.strip()

    if not user_text:
        print("Whisper returned empty text.")
        continue


    print("\nUser:")
    print(user_text)


    # ========================================================
    # BERT - Should AI Respond?
    # ========================================================

    inputs = tokenizer(
        user_text,
        return_tensors="np",
        truncation=True
    )

    onnx_inputs = {
        "input_ids": inputs["input_ids"],
        "attention_mask": inputs["attention_mask"]
    }

    logits = int8_session.run(
        None,
        onnx_inputs
    )[0]

    probs = softmax(logits)

    prediction = np.argmax(
        probs,
        axis=1
    )[0]

    respond_probability = probs[0][SHOULD_RESPOND_CLASS]

    print(
        f"Should respond probability: "
        f"{respond_probability:.3f}"
    )


    # ========================================================
    # Decision
    # ========================================================

    if prediction != SHOULD_RESPOND_CLASS:

        print("BERT decided: DON'T RESPOND")
        continue


    print("BERT decided: RESPOND")


    # ========================================================
    # LLM
    # ========================================================

    stream = chat(
        model=LLM_MODEL,
        messages=[
            {
                "role":"system",
                "content":"You are helpfull AI Assistent, for given text or question you will respond in short like normal person, give long response only if asked or needed"
            },
            {
                "role": "user",
                "content": user_text
            }
            ],
            stream=True
    )

    print("\nLLM:")
    audio_streamer = sd.OutputStream(
            samplerate=24000,
            channels=1,
            dtype="float32",
            blocksize=2048
        )
    audio_streamer.start()
    try:
        for chunk in stream:
            text = chunk["message"]["content"]
            print(text, end="", flush=True)
    
            generator = pipeline(text, voice=VOICE)
    
            for _, _, audio in generator:
                audio_streamer.write(audio)

    finally:
        audio_streamer.stop()     
        audio_streamer.close()
        
    

Using cache found in /home/keerthivardhan/.cache/torch/hub/snakers4_silero-vad_master


Loaded tokenizer
Loaded INT8 BERT classification model
Listening...
Speak now.

Speech detected...
Maximum utterance duration reached.
Calling Whisper...

User:
I have continued my work on building autonomous local LLM. I have integrated all the components starting from VAD, ASL and the text-to-speech model. And right now I am testing all the things. But one thing I have noticed in the experiment was the Queen 2.5 3 billion model sometimes responding garbage but
Should respond probability: 0.695
BERT decided: RESPOND

LLM:
It sounds like you're making significant progress with building an autonomous local language model! That's fantastic. When you mentioned the Queen 2.5 billion model occasionally producing "garbage" responses, that could be due to several reasons:

1. **Model Overconfidence or Underfitting**: The model might have overconfidently predicted what it thinks is a response when it's actually not suitable for the context.

2. **Training Data

KeyboardInterrupt: 

### observation

1. Pipelie is working all the compenents are working fine and can here the output
2. The output is not smooth, it is taking some time to output audio word

### Next Step

1. **Multithreading and shared memory**

## Imports and configurations

In [2]:
import torch
import sounddevice as sd
import numpy as np
from scipy.io.wavfile import write
from pathlib import Path
import subprocess
import time
from transformers import AutoTokenizer
import onnxruntime as ort
from ollama import chat
from kokoro import KPipeline
from IPython.display import Audio, display
import soundfile as sf
from queue import Queue
import threading
# ============================================================
# Configuration
# ============================================================

SAMPLE_RATE = 16000
CHUNK_SIZE = 512

SPEECH_THRESHOLD = 0.5
SILENCE_DURATION = 1.0
MAX_UTTERANCE_SECONDS = 30

LLM_MODEL = "qwen2.5-coder:3b"

WHISPER_DIR = Path("../whisper.cpp")
WHISPER_MODEL = WHISPER_DIR / "models" / "ggml-base.en.bin"
WHISPER_CLI = WHISPER_DIR / "build" / "bin" / "whisper-cli"

AUDIO_FILE = Path("utterance.wav")

SHOULD_RESPOND_CLASS = 1

VOICE = "af_heart"


### Loading models

In [29]:

# ============================================================
# Load Should AI Respond model
# ============================================================

model, utils = torch.hub.load(
    repo_or_dir="snakers4/silero-vad",
    model="silero_vad",
    trust_repo=True
)

tokenizer = AutoTokenizer.from_pretrained(
    "./should_ai_respond_model"
)

print("Loaded tokenizer")


int8_session = ort.InferenceSession(
    "./should_ai_respond_int8.onnx",
    providers=["CPUExecutionProvider"]
)

print("Loaded INT8 BERT classification model")

pipeline = KPipeline(lang_code="a")
audio_streamer = sd.OutputStream(
            samplerate=24000,
            channels=1,
            dtype="float32",
            blocksize=2048
        )

Using cache found in /home/keerthivardhan/.cache/torch/hub/snakers4_silero-vad_master


Loaded tokenizer
Loaded INT8 BERT classification model


In [4]:
### Helper
# ============================================================
# Helper
# ============================================================

def softmax(x):
    exp_x = np.exp(
        x - np.max(x, axis=1, keepdims=True)
    )
    return exp_x / exp_x.sum(
        axis=1,
        keepdims=True
    )


### Listener Worker (VAD + ASR)

In [15]:
def listen_for_speech():
    audio_chunks = []
    speech_started = False
    silence_start = None
    speech_start_time = time.time()

    model.reset_states()

    with sd.InputStream(
        samplerate=SAMPLE_RATE,
        channels=1,
        dtype="float32",
        blocksize=CHUNK_SIZE
    ) as stream:

        while True:

            chunk, overflowed = stream.read(CHUNK_SIZE)
            chunk = chunk[:, 0]

            chunk_tensor = torch.from_numpy(chunk)

            speech_probability = model(
                chunk_tensor,
                SAMPLE_RATE
            ).item()

            is_speech = speech_probability >= SPEECH_THRESHOLD

            # User is speaking
            if is_speech:

                if not speech_started:
                    print("Speech detected...")
                    speech_started = True

                audio_chunks.append(chunk.copy())
                silence_start = None

            # User is silent
            else:

                if speech_started:

                    audio_chunks.append(chunk.copy())

                    if silence_start is None:
                        silence_start = time.time()

                    silence_time = time.time() - silence_start

                    if silence_time >= SILENCE_DURATION:
                        print("User finished speaking.")
                        break

            # Maximum speech duration
            if time.time() - speech_start_time >= MAX_UTTERANCE_SECONDS:
                print("Maximum speech duration reached.")
                break

    if not speech_started:
        print("No speech detected.")
        return None

    audio = np.concatenate(audio_chunks)

    write(
        AUDIO_FILE,
        SAMPLE_RATE,
        audio
    )

    return AUDIO_FILE

In [19]:
def Listener(Listener_BERT_shared_queue, stop_event):

    while not stop_event.is_set():

        audio_file = listen_for_speech()

        if audio_file is None:
            continue    # Keep listening

        print("Calling Whisper...")

        try:
            result = subprocess.run(
                [
                    str(WHISPER_CLI),
                    "-m", str(WHISPER_MODEL),
                    "-f", str(audio_file),
                    "-nt"
                ],
                capture_output=True,
                text=True,
                check=True
            )

        except subprocess.CalledProcessError as e:
            print("Whisper failed:")
            print(e.stderr)
            continue    # Listen again

        user_text = result.stdout.strip()

        if not user_text:
            print("Whisper returned empty text.")
            continue    # Listen again

        print("\nUser:")
        print(user_text)

        Listener_BERT_shared_queue.put(user_text)

    print("Listener: came out of while loop")

### BERT


In [7]:
def ShouldAIRespond(Listener_BERT_shared_queue, tokenizer,int8_session, softmax, LLM_BERT_shared_queue, stop_event):
    while not stop_event.is_set():
        '''
        If Listener_BERT_shared_queue is empty or less then 3 elements wait for it to fill for 1 sec
        if even after 1 sec it same size , then continue
        '''
        user_text = Listener_BERT_shared_queue.get()
        inputs = tokenizer(
            user_text,
            return_tensors="np",
            truncation=True
        )
    
        onnx_inputs = {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"]
        }
    
        logits = int8_session.run(
            None,
            onnx_inputs
        )[0]
    
        probs = softmax(logits)
    
        prediction = np.argmax(
            probs,
            axis=1
        )[0]
    
        respond_probability = probs[0][SHOULD_RESPOND_CLASS]
    
        print(
            f"Should respond probability: "
            f"{respond_probability:.3f}"
        )
    
    
        # ========================================================
        # Decision
        # ========================================================
    
        if prediction != SHOULD_RESPOND_CLASS:
    
            print("BERT decided: DON'T RESPOND")
            continue
    
    
        print("BERT decided: RESPOND")
        LLM_BERT_shared_queue.put(user_text)

    print("BERT: Came out of while loop")


### LLM

In [25]:
import re
def LLM(LLM_MODEL, LLM_BERT_shared_queue, LLM_Kokoro_shared_queue,stop_event):
    while not stop_event.is_set():
        '''
        if LLM_BERT_shared_queue is empty wait and allow other threads to use the cpu 
        '''
        user_text = LLM_BERT_shared_queue.get()
        stream = chat(
            model=LLM_MODEL,
            messages=[
                {
                    "role":"system",
                    "content":"You are helpfull AI Assistent, for given user message you will respond in short with in 5 to 6 lines like normal person,"
                },
                {
                    "role": "user",
                    "content": user_text
                }
                ],
            stream=True
        )

        # for chunk in stream:
        #     text = chunk["message"]["content"]
        #     print(text, end="", flush=True)
        #     cleaned_text = re.sub(r"[\r\n]+", " ", text)
        #     cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()
        #     LLM_Kokoro_shared_queue.put(cleaned_text)
        buffer = ""

        for chunk in stream:
            token = chunk["message"]["content"]
            print(token, end="", flush=True)
        
            buffer += token
        
            if buffer.endswith((".", "!", "?")):
                LLM_Kokoro_shared_queue.put(buffer.strip())
                buffer = ""
        
        # Remaining text
        if buffer.strip():
            LLM_Kokoro_shared_queue.put(buffer.strip())
            
    print("LLM: came out of while loop")
        
        

In [28]:
from queue import Empty

def Assistant(audio_streamer, pipeline, LLM_Kokoro_shared_queue, stop_event):

    audio_streamer.start()

    while not stop_event.is_set():

        try:
            text = LLM_Kokoro_shared_queue.get(timeout=1)

        except Empty:
            # Nothing to speak right now.
            continue

        generator = pipeline(text, voice=VOICE)

        for _, _, audio in generator:
            audio_streamer.write(audio)

    audio_streamer.stop()
    audio_streamer.close()

    print("Assistant: came out of while loop")

In [30]:
def main():

    # Shared queues
    Listener_BERT_shared_queue = Queue()
    LLM_BERT_shared_queue = Queue()
    LLM_Kokoro_shared_queue = Queue()

    # Stop signal
    stop_event = threading.Event()

    # Workers
    Listener_worker = threading.Thread(
        target=Listener,
        args=(Listener_BERT_shared_queue, stop_event)
    )

    ShouldAIRespond_worker = threading.Thread(
        target=ShouldAIRespond,
        args=(
            Listener_BERT_shared_queue,
            tokenizer,
            int8_session,
            softmax,
            LLM_BERT_shared_queue,
            stop_event
        )
    )

    LLM_worker = threading.Thread(
        target=LLM,
        args=(
            LLM_MODEL,
            LLM_BERT_shared_queue,
            LLM_Kokoro_shared_queue,
            stop_event
        )
    )

    Assistant_worker = threading.Thread(
        target=Assistant,
        args=(
            audio_streamer,
            pipeline,
            LLM_Kokoro_shared_queue,
            stop_event
        )
    )

    # Start workers
    Listener_worker.start()
    ShouldAIRespond_worker.start()
    LLM_worker.start()
    Assistant_worker.start()

    print("All workers have been started.")

    try:
        
        while True:
            time.sleep(1)

    except KeyboardInterrupt:
        print("\nStopping assistant...")
        stop_event.set()

    # Wait for workers to finish
    Listener_worker.join()
    ShouldAIRespond_worker.join()
    LLM_worker.join()
    Assistant_worker.join()

    print("Assistant stopped.")

In [ ]:
main()

All workers have been started.
Speech detected...
User finished speaking.
Calling Whisper...

User:
what is L L M?
Should respond probability: 0.631
BERT decided: RESPOND
L.L. Motley is an American actor known for his roles in movies like "Inception" and "The Dark Knight."Speech detected...
User finished speaking.
Calling Whisper...

User:
What is the LLM?
Should respond probability: 0.887
BERT decided: RESPOND
LLM stands for Large Language Model. It's a type of artificial intelligence that can understand and generate human language. Imagine it as a super smart computer that learns from lots of text data to create amazing responses!Maximum speech duration reached.
No speech detected.
Speech detected...
User finished speaking.
Calling Whisper...

User:
What are you doing?
Should respond probability: 0.661
BERT decided: RESPOND
Just here to chat and answer your questions. How can I assist you today?Speech detected...
User finished speaking.
Calling Whisper...

User:
These are multi-threa